In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GlobalAveragePooling1D,
    Dense
)

I0000 00:00:1787572949.805018  216156 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787572949.934680  216156 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787572953.749787  216156 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Load the Preprocessed Data

In [2]:
X_train_padded = np.load(
    "../Dataset/processed/X_train_padded.npy"
)

X_val_padded = np.load(
    "../Dataset/processed/X_val_padded.npy"
)

X_test_padded = np.load(
    "../Dataset/processed/X_test_padded.npy"
)

y_train = np.load(
    "../Dataset/processed/y_train.npy"
)

y_val = np.load(
    "../Dataset/processed/y_val.npy"
)

y_test = np.load(
    "../Dataset/processed/y_test.npy"
)

In [3]:
print("Training:", X_train_padded.shape)
print("Validation:", X_val_padded.shape)
print("Testing:", X_test_padded.shape)

Training: (34705, 200)
Validation: (7439, 200)
Testing: (7438, 200)


# Load the Tokenizer

In [4]:
with open(
    "../Dataset/processed/tokenizer.pkl",
    "rb"
) as file:

    tokenizer = pickle.load(file)

In [5]:
with open(
    "../Dataset/processed/preprocessing_config.pkl",
    "rb"
) as file:

    preprocessing_config = pickle.load(file)

In [6]:
MAX_SEQUENCE_LENGTH = preprocessing_config[
    "max_sequence_length"
]

NUM_WORDS = preprocessing_config[
    "num_words"
]

print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Vocabulary limit:", NUM_WORDS)

Maximum sequence length: 200
Vocabulary limit: 20000


In [7]:
# Integer Encoding

word_index = tokenizer.word_index

print("Vocabulary size:", len(word_index))

print("\nFirst 20 words:")
print(list(word_index.items())[:20])

Vocabulary size: 85872

First 20 words:
[('<OOV>', 1), ('movie', 2), ('film', 3), ('not', 4), ('one', 5), ('like', 6), ('good', 7), ('no', 8), ('time', 9), ('even', 10), ('would', 11), ('story', 12), ('really', 13), ('see', 14), ('well', 15), ('much', 16), ('bad', 17), ('get', 18), ('great', 19), ('people', 20)]


In [8]:
with open(
    "../Dataset/processed/text_splits.pkl",
    "rb"
) as file:

    text_splits = pickle.load(file)

In [9]:
X_train = text_splits["X_train"]
X_val = text_splits["X_val"]
X_test = text_splits["X_test"]

In [10]:
train_text = X_train
val_text = X_val
test_text = X_test

In [11]:
# TF-IDF Vectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [12]:
# Fit TF-IDF Only on Training Data

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_text
)


X_val_tfidf = tfidf_vectorizer.transform(
    val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_text
)

In [13]:
# check TF-ID shapes

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

TF-IDF training shape: (34705, 20000)
TF-IDF validation shape: (7439, 20000)
TF-IDF testing shape: (7438, 20000)


In [14]:
# Inspect TF-ID Features

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFirst 20 features:")
print(feature_names[:20])

sample_vector = X_train_tfidf[0]

print(sample_vector)

Number of features: 20000

First 20 features:
['aaron' 'abandon' 'abandoned' 'abbey' 'abbot' 'abbott' 'abbott costello'
 'abby' 'abc' 'abducted' 'abilities' 'ability' 'able' 'able find'
 'able get' 'able keep' 'able make' 'able see' 'able watch' 'ably']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 105 stored elements and shape (1, 20000)>
  Coords	Values
  (0, 12095)	0.06905252313148046
  (0, 11521)	0.10336760424477463
  (0, 15406)	0.04181268597801286
  (0, 12541)	0.047340163575216296
  (0, 5417)	0.06366210034697742
  (0, 13938)	0.050323302509996184
  (0, 19692)	0.05210128899315189
  (0, 4743)	0.05932049864354727
  (0, 19988)	0.08296951997241968
  (0, 11467)	0.03843911384868589
  (0, 10558)	0.049038859770200845
  (0, 6904)	0.03515293323060653
  (0, 19175)	0.038311930053506534
  (0, 761)	0.044733051872236106
  (0, 14946)	0.11565688167065921
  (0, 19992)	0.2625742472376709
  (0, 11022)	0.09471488791800416
  (0, 16818)	0.11370020512071431
  (0, 8893)	0.05742037518464474
 

In [15]:
with open(
    "../Dataset/processed/tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(tfidf_vectorizer, file)

In [16]:
# TF-IDF Representation Summary

# Word Embeddinds

vocab_size = min(
    NUM_WORDS,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size used by model:", vocab_size)

Vocabulary size used by model: 20000


In [17]:
# Create a Trainable Embedding Layer

EMBEDDING_DIM = 128

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH
)

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Text_Sentiment_Analysis-D79r_KZq/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
# Build a Small Embedding Demonstration Model

embedding_demo_model = Sequential([
    embedding_layer,
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

embedding_demo_model = Sequential([
    embedding_layer,
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [19]:
embedding_demo_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Save Feature-Representation Information

feature_config = {
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "tfidf_max_features": 20000,
    "tfidf_ngram_range": (1, 2)
}

In [21]:
with open(
    "../Dataset/processed/feature_config.pkl",
    "wb"
) as file:

    pickle.dump(feature_config, file)